# Project 7: Customer Churn — Classification in Practice

**Dataset:** Telco Customer Churn (Kaggle, ~7,000 customers)

**Goal:** Predict which customers will leave (churn) using real telecom data. Learn about confusion matrices, ROC curves, and the Precision-Recall trade-off.

In [ ]:
# ====== Setup: imports + data loader ======
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

DATA_URL = 'https://raw.githubusercontent.com/lucyyuhongxia-gmail/A4_ML_Projects/main/datasets'
def load_data(fn):
    """Load a real dataset from GitHub."""
    df = pd.read_csv(f"{DATA_URL}/{fn}")
    print(f"Loaded: {fn} -> {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df
print('Setup OK!')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix,
    ConfusionMatrixDisplay)

## 1. Load and Prepare Real Telecom Data

This dataset contains ~7,000 real customers from a US telecom company. Each row has customer demographics, account information, and services used. The target is: did this customer churn (leave)?

In [ ]:
df = load_data('telco_churn.csv')
print(f'Shape: {df.shape}')
print(f'Churn rate: {(df.Churn == "Yes").mean():.1%}')

# Clean numeric column — 清理数值列(有些值可能非数字)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])

# Create target — 创建目标变量: 1=流失, 0=留下
df['Churn_Label'] = (df['Churn'] == 'Yes').astype(int)

# Encode categorical features — 类别特征编码
# 机器学习模型只能处理数字, 需要把文本标签转为数字
cat_cols = df.select_dtypes(include=['object']).columns.drop(['customerID','Churn'])
for col in cat_cols:
    df[col + '_enc'] = LabelEncoder().fit_transform(df[col].astype(str))

features = ['tenure','MonthlyCharges','TotalCharges'] + [c+'_enc' for c in cat_cols]
X = df[features]; y = df['Churn_Label']
print(f'Feature matrix: {len(features)} features, {len(X):,} samples')

## 2. Train and Compare Models

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# K-NN (k=7)
knn = KNeighborsClassifier(n_neighbors=7).fit(X_train_s, y_train)
y_knn = knn.predict(X_test_s)
y_knn_prob = knn.predict_proba(X_test_s)[:, 1]

# Decision Tree (depth-limited for interpretability)
dt = DecisionTreeClassifier(
    max_depth=5, min_samples_leaf=50, random_state=42)
dt.fit(X_train, y_train)
y_dt = dt.predict(X_test)
y_dt_prob = dt.predict_proba(X_test)[:, 1]

# Comparison table
comp = pd.DataFrame({
    'Metric': ['Accuracy','Precision','Recall','F1','ROC AUC'],
    'K-NN': [accuracy_score(y_test, y_knn),
             precision_score(y_test, y_knn),
             recall_score(y_test, y_knn),
             f1_score(y_test, y_knn),
             roc_auc_score(y_test, y_knn_prob)],
    'D-Tree': [accuracy_score(y_test, y_dt),
               precision_score(y_test, y_dt),
               recall_score(y_test, y_dt),
               f1_score(y_test, y_dt),
               roc_auc_score(y_test, y_dt_prob)],
}).round(4)
print(comp.to_string(index=False))

## 3. Visualize: Confusion Matrices & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_knn, ax=axes[0], cmap='Blues',
    display_labels=['Stayed','Churned'])
axes[0].set_title('K-NN Confusion Matrix')
ConfusionMatrixDisplay.from_predictions(
    y_test, y_dt, ax=axes[1], cmap='Blues',
    display_labels=['Stayed','Churned'])
axes[1].set_title('Decision Tree Confusion Matrix')
plt.tight_layout(); plt.show()

# Feature importance (decision tree only)
fi = pd.DataFrame({'Feature': features, 'Importance': dt.feature_importances_})
fi = fi.sort_values('Importance', ascending=False)
print('\nTop 10 Features by Importance:')
print(fi.head(10).to_string(index=False))
print()
print('💡 Key business insight: tenure is the #1 predictor of churn!')
print('   Customers who stay longer are less likely to churn.')

## 4. ROC Curve Analysis

In [ ]:
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_knn_prob)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_dt_prob)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr_knn, tpr_knn, label=f'K-NN (AUC={roc_auc_score(y_test, y_knn_prob):.3f})')
ax.plot(fpr_dt, tpr_dt,  label=f'DT (AUC={roc_auc_score(y_test, y_dt_prob):.3f})')
ax.plot([0,1], [0,1], 'k--', label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Churn Prediction')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print('Reading ROC AUC:')
print('  1.0 = perfect classifier')
print('  0.9 = excellent')
print('  0.8 = good ✓ (our Decision Tree)')
print('  0.5 = random guessing')

## Check Your Understanding

1. **Baseline accuracy:** If you predicted "Stayed" for every customer, what accuracy would you get? Is our model better than this baseline?

2. **Precision vs Recall:** The telecom company has budget to call only 10% of customers for retention offers. Should they optimize for Precision or Recall? Why?

3. **Business insight:** Decision Tree says "tenure" is the most important feature. What business action would you take based on this? (Hint: think about loyalty programs, early engagement...)

4. **ROC AUC:** Our Decision Tree has AUC≈0.82. Is this good enough for production? What industry standards would you compare against?